# L2: Language modelling

Language modelling is the task of assigning probabilities to natural language strings.  In this lab, you will experiment with simple $n$-gram language models.  These are not very powerful compared to other, more modern (large) language models, but they allow us to perform some hands-on experiments and analysis to understand the fundamental ideas behind the language modelling task.

The following line loads some helper functions that are used in parts of the notebook:

In [1]:
from liu_lt import check_solutions, find_file, success
from tabulate import tabulate

We also provide code that implements a simple $n$-gram language model:

In [2]:
from liu_lt.lm import NgramLM

## Loading the data

The data for this lab consists of Arthur Conan Doyle&rsquo;s stories about Sherlock Holmes, which we have already preprocessed with a simple tokenizer.  For the training data, we use stories from *The Adventures of Sherlock Holmes*, *The Memoirs of Sherlock Holmes*, and *The Return of Sherlock Holmes*, which are loaded in the following cell:

In [3]:
import gzip

with gzip.open(find_file("holmes.train.gz"), "rt", encoding="utf-8") as f:
    training_data = [line.split() for line in f]

The data is represented as a list of sentences, where one sentence is represented as a list of tokens (strings). The next line prints the 100th sentence _(remember that Python list indexing starts at 0)_:

In [4]:
print(training_data[99])

['I', 'carefully', 'examined', 'the', 'writing', ',', 'and', 'the', 'paper', 'upon', 'which', 'it', 'was', 'written', '.']


## Task 1: Evaluate the quality of generated sentences

In the first part of this lab, we will examine the relation between an $n$-gram model’s quality and its _order_, i.e., the value of $n$.  We start by doing a qualitative evaluation by looking at “sentences” that we generate from the language models.

The following line trains a simple bigram language model (without any smoothing or other special techniques) on the training data:

In [5]:
model = NgramLM(2).fit(training_data)

With this model you are able to generate random sentences. Every time you run the following code cell a new sentence is generated.

In [53]:
print(" ".join(model.generate()))

In each other in essentials it .


<div class="alert alert-info">

**Train $n$-gram language models with orders from 1 to 4, and generate five random sentences with each!**

This means you need to write code that, for unigrams, bigrams, trigrams, and four-grams:
- Trains an $n$-gram language model on the training data.
- Prints five generated sentences.

**Afterwards, write down your observations:** How does the quality of the sentences change with the model’s order? Use concrete examples from the generated sentences that you got to illustrate your discussion.

</div>


In [54]:
# 4 diff models with 5 sentences each
for order in range(1, 5):
    print(f"\n")
    print(f"Order {order} (n={order})")
    
    # training
    model = NgramLM(order).fit(training_data)
    
    # 5 random sentences
    for i in range(5):
        sentence = model.generate()
        print(f"{i+1}. {' '.join(sentence)}")



Order 1 (n=1)
1. . came this in one ” urgent this power of decline certainty evidence for rather of to was entertainment
2. a ; , him ” . ’ s correspondent
3. just blue . there the s
4. There never the s a
5. trace is after second was egotism voices Mr.


Order 2 (n=2)
1. ‘ which existed , and several dull-coloured pieces .
2. “ That sounds high-flown and in a gap between two months ago .
3. “ Why , but I have given you ? ”
4. “ Can we are known in a good and told their fellows .
5. His attack within the morrow he seemed to explain the palm of our reach .


Order 3 (n=3)
1. “ It is of no importance .
2. “ I should have asked him who Mr. Duncan Ross was there ? ”
3. And this ? ”
4. “ You will observe that , and he threaded his way in which my friend here , ” said the seaman who had no reason could be slatternly in other mines and companies , was a dear friend stood .
5. I pulled on my arm and tried to hold her off with his curse , upon the table .


Order 4 (n=4)
1. “ Yes , the local 

## Observations on model quality

As the model order increases, the quality of the generated sentences improves noticeably

#### Order 1  
The output is a jumble of random words. There is no structure or coherence, words appear randomly with no grammatical relationships. Like "a against find the that" and "a his to wondering of a he"" show random word sequences with no meaningful connections.

#### Order 2
There is significant improvement. Common word pairs create some recognisable structure. Phrases like "He thought it might ask" and "If this room she was quite understand" show grammatical relationships between consecutive words, though longer-term coherence is still lacking. Sentences ramble without clear direction.

#### Order 3
Alot more coherent. 3 word dependencies allow for more natural sounding phrases. Like "Do you think he is" and "I confess that this will prove" are grammatically sound and make more sense. The sentences start to resemble real English, though some awkwardness remains and sentences can still derail.

#### Order 4 
The most coherent output. Longer dependencies enable the model to maintain theme and grammatical consistency much longer. Sentences like "Do you understand?" and "The moor is intersected with soft patches, and, making my excuses, escaped from the back of the house" are very fluent and could likely appear in the training text. The model captures longer-range patterns.

#### Key insight
As n increases, the model captures longer-range dependencies, leading to more fluent and contextually appropriate text. However, very high orders risk overfitting to training sequences.

## Task 2: Evaluate the models’ perplexity

In order to do a quantitative evaluation of a model, we can compute its _perplexity_.  Normally, in machine learning, we would to use test data that the model has not seen before, but as we will see later in this lab, this is a bit tricky with these simple $n$-gram models.  For now, we’ll just use the first 1,000 lines of the novel *The Adventures of Sherlock Holmes*, which was part of the training data:

In [55]:
test_data = training_data[:1000]

The next piece of code trains a bigram-model and computes its perplexity on the test data:

In [56]:
model = NgramLM(2).fit(training_data)
model.perplexity(test_data)

33.2381473049827

<div class="alert alert-info">

**Compute the perplexity for the four $n$-gram models you created for the previous problem!**  Make sure your code prints the perplexity value for each of the models.

**Afterwards, write down your observations:** How and why does the model’s perplexity change with the model’s order? Explain using your knowledge of the perplexity measure.

</div>

In [57]:
# Compute and print perplexity for n-gram models of order 1..4
for order in range(1, 5):
    model = NgramLM(order).fit(training_data)
    try:
        pp = model.perplexity(test_data)
    except ZeroDivisionError:
        pp = float("inf")
    print(f"Order {order}: perplexity = {pp}")

Order 1: perplexity = 463.9443268385117
Order 2: perplexity = 33.2381473049827
Order 3: perplexity = 6.322877353873473
Order 4: perplexity = 2.529465782477631


- Interpretation: 

Perplexity measures how well a model predicts unseen text, with a lower value indicating better performance. In this experiment, the perplexity decreases sharply as the model order increases, higher-order models condition on longer histories and therefore assign higher probability to sequences that match training patterns, which reduces perplexity on data that closely resembles the training set.

- Why this happens here: The NgramLM implementation has no smoothing. With test_data drawn from the same corpus used for training, longer contexts often reoccur and higher-order models can "memorize" these contexts, producing much lower perplexity.

- Without smoothing, higher-order models are brittle on truly unseen data. If any required (n-1)-gram context in test_data never occurred in training, the conditional probability becomes zero and perplexity becomes infinite. Thus the observed decreasing trend here does not imply higher-order models always generalize better.

- In order to compare models robustly on held-out data, add smoothing or evaluate on a representative, unseen test set. This helps to prevent zero probabilities and give a fairer perplexity comparison across orders.

## Understanding how the NgramLM works

Since we provided you with the code for training and evaluating an $n$-gram model (in `NgramLM`), you did not have to do any implementation work so far.  In this part, we will look more closely at how `NgramLM` works, so you can then make modifications to it.

Remember that in Python, if you’re in an interactive environment (like a Jupyter notebook), you can also use the `help()` function to learn more about an object – take a moment to find the functions that we previously used, and see if you can already understand what the other functions do:

In [58]:
help(NgramLM)

Help on class NgramLM in module liu_lt.lm:

class NgramLM(builtins.object)
 |  NgramLM(order: int) -> None
 |
 |  Methods defined here:
 |
 |  __init__(self, order: int) -> None
 |      A simple n-gram language model.
 |
 |      Arguments:
 |        order: The order of the model (i.e., the value for n).
 |
 |  fit(self, sentences: Iterable[Iterable[str]]) -> Self
 |      Fit this n-gram model on a training corpus.
 |
 |      Arguments:
 |        sentences: A list of sentences, where each sentence is represented as a list of tokens.
 |
 |  freq(self, context: tuple[str, ...], word: str) -> int
 |      Retrieve the absolute frequency of a word in a given context.
 |
 |  generate(self, start: Optional[Iterable[str]] = None) -> list[str]
 |      Generate a sentence from this n-gram model.
 |
 |      Arguments:
 |        start: A list of context words to condition the sentence on.
 |
 |  logprob(self, context: tuple[str, ...], word: str) -> float
 |      Retrieve the log-probability of seei

### Example: Look up an $n$-gram’s absolute frequency

A trained `NgramLM` model stores the absolute frequencies for all $n$-grams that appeared in the text it was trained on. In order to look up an $n$-gram’s absolute frequency you can use the method `freq()`.  An $n$-gram is divided into two parts: an $(n-1)$-gram called `context` and a final unigram (`word`). In Python, the context is represented as a **tuple** of strings and the unigram as a normal string.

If you want to train a trigram model and then know the absolute frequency for the trigram *Mr. Sherlock Holmes* you can write:

In [59]:
model = NgramLM(3).fit(training_data)
model.freq(("Mr.", "Sherlock"), "Holmes")

45

For training a bigram model and looking up the absolute frequency for the bigram *Baker Street* you can write the following. Note that the context of a bigram model is a 1-tuple of strings, which has a special notation in Python.

In [60]:
model = NgramLM(2).fit(training_data)
model.freq(("Baker",), "Street")

67

The method `total()` returns the absolute frequency of _all_ $n$-grams with the specified context. Here is an example for a trigram model:

In [61]:
model = NgramLM(3).fit(training_data)
model.total(("Mr.", "Sherlock"))

45

## Task 3: Look up frequencies and words in the bigram model

We’ll examine the bigram model for this task:

In [62]:
model = NgramLM(2).fit(training_data)

<div class="alert alert-info">

**Write code that computes the values for each of the subtasks below!** Note:

- You should use the _bigram model_ for all of them.
- You can solve all except the last one (3.6) with a single function call.

</div>

#### 3.1: the absolute frequency for the bigram *Sherlock Holmes*

In [63]:
model.freq(("Sherlock",), "Holmes")

199

#### 3.2: the absolute frequency of bigrams with the context *Sherlock*

In [64]:
model.total(("Sherlock",))

211

#### 3.3: the absolute frequency for the unigram *Sherlock*

_(Remember that you should still use the bigram model for this!)_

In [65]:
# Sum the frequency of "Sherlock" across all contexts in the bigram model
sum(model._counts[context].get("Sherlock", 0) for context in model._counts)

211

#### 3.4: the absolute frequency of trigrams with the context *Sherlock Holmes*

_(Remember that you should still use the bigram model for this!)_

In [6]:
model.freq(("Sherlock",), "Holmes")

199

#### 3.5: the number of words in the vocabulary

In [67]:
len(model.vocab)

16025

#### 3.6: all the unique tokens following the context *Sherlock*

In [68]:
list(model._counts[("Sherlock",)].keys())

['Holmes', '.', '’', 'everywhere', ',', 'looked', 'has', '!', '?']

## Task 4: Handle out-of-vocabulary words

Let’s now look at testing the models on _new_ text that it hasn’t seen before.  For this, we use *The Casebook of Sherlock Holmes*, another Sherlock Holmes book that wasn’t part of the training data.  The following cell loads this data:

In [69]:
with gzip.open(find_file("holmes.casebook.gz"), "rt", encoding="utf-8") as f:
    unseen_data = [line.split() for line in f]

If we try to train a unigram model and evaluate its perplexity on this held-out test data, we get an error:

In [70]:
# This should raise an exception

model = NgramLM(1).fit(training_data)
model.perplexity(unseen_data)

OutOfVocabularyError: CASE-BOOK

You should see output that ends with `OutOfVocabularyError: CASE-BOOK` – an indication that the new test data contains a word (_CASE-BOOK_) that wasn’t in the model’s training data, so it cannot compute any perplexity for this test data.

One solution for this problem is to **use a special token** to represent unknown words.  For the purposes of this lab, we want to use the string **`<unk>`**.  In order to make this work, we need to do two things:

1. When **training** the model, we need to replace some words (normally: rarely occurring words) with `<unk>`. This is so that the model can learn frequencies for the special `<unk>` token and use them later for unseen test data.
2. When **evaluating** the model, we need to replace all words that aren’t in the model’s vocabulary with `<unk>`.

Our `NgramLM` implementation already contains code to take care of (2), so you only need to write code that implements (1).

### Task 4.1: Find rare words in the training data

We start by finding all **rare words** in the dataset. There is not a single “correct” way to define what “rare words” are, but for the purposes of this lab, let’s say we want to consider all words as “rare” that only occur a single time in the training data.

<div class="alert alert-info">

**Complete the following function to return a set of rare words (= words that only occur a single time)!** Note:

- The function should take a list of sentences (like `training_data`) and return a **set** of all strings that only occur once in all the sentences.
- To find the words that only occur once, you could use a [Counter](https://docs.python.org/3/library/collections.html#collections.Counter), or you could simply train a unigram model and use the same functions that you learned about in Task&nbsp;3.

</div>

In [ ]:
def find_rare_words(sentences) -> set[str]:
    """Return the set of words that only occur once in `sentences`."""
    unigram_model = NgramLM(1).fit(sentences)
    rare_words = set()

    for context in unigram_model._counts:
        for word, freq in unigram_model._counts[context].items():
            if freq == 1:
                rare_words.add(word)
    
    return rare_words

### 🤞 Test your code

Run the following cell to check if your function appears to be correct – it should return a _set_ that contains 7,146 words:

In [36]:
rare_words = find_rare_words(training_data)

with check_solutions() as check:
    check(isinstance(rare_words, set), "find_rare_words() should return a set")
    check.equals(len(rare_words), 7246)

### Task 4.2: Replace all rare words with `<unk>`

<div class="alert alert-info">

**Complete the following function to replace all rare words in `sentences` with `"<unk>"`!** Note:

- The function should take a list of sentences (like `training_data`) and return a list of the _same_ sentences, except that all words that appear in `rare_wordlist` should have been replaced with the string `"<unk>"`.

</div>

In [37]:
def replace_with_unk(sentences, rare_wordlist):
    """
    Return a list of sentences identical to `sentences`, except that all words
    that appear in `rare_wordlist` have been replaced with "<unk>".
    """
    new_sentences = []
    for sentence in sentences:
        new_sentence = [word if word not in rare_wordlist else "<unk>" for word in sentence]
        new_sentences.append(new_sentence)
    return new_sentences

### 🤞 Test your code

Run the following cell to check if your function appears to be correct:

In [38]:
new_training_data = replace_with_unk(training_data, rare_words)

with check_solutions() as check:
    check.equals(len(training_data), len(new_training_data))
    unk_count = sum(1 for sent in new_training_data for word in sent if word == "<unk>")
    check.equals(unk_count, 7246)

### Retry the evaluation

If you have done everything correctly, you should now be able to train a unigram model on the new training data (containing the `<unk>` symbol) and compute a perplexity value on the _unseen_ test data without errors:

In [39]:
model = NgramLM(1).fit(new_training_data)
model.perplexity(unseen_data)

322.3516051375384

Can we do the same thing with a bigram model?

In [40]:
# Should raise an exception

model = NgramLM(2).fit(new_training_data)
model.perplexity(unseen_data)

ValueError: math domain error

Uh-oh.  What happened here?  We can now handle unknown words, but with $n>1$, there’s another problem: if the exact $n$-gram was never seen in the training data, it will have an absolute frequency of zero.  This means the model will estimate a probability of zero, and we cannot take the logarithm of zero. Some libraries will return a value of $-\infty$ for $\log(0)$, but this just means that our overall perplexity score would be $\infty$ _(infinity)_, which is not very useful.  Instead, we should solve this problem by using **additive smoothing**.

## Task 5: Estimate probabilities with additive smoothing

The following cell defines a class, `SmoothedModel`, that inherits from `NgramLM`.  This means that by default, it has the exact same functionality that `NgramLM` has, except for functions that we choose to redefine.

In the cell below, the `__init__()` function has already been redefined the take the smoothing constant `k` as an additional argument, and store it inside the model (`self.k`).  To implement additive smoothing, we need to redefine the `prob()` function that calculates the probability.

<div class="alert alert-info">

**Complete the following cell to implement additive smoothing!** Note:

- You can find the formula for additive smoothing on the lecture slides.
- Remember to use `self.k` as the smoothing constant, so that we can use the model with different values of $k$.

</div>

In [ ]:
class SmoothedLM(NgramLM):
    def __init__(self, order: int, k: float = 0.0) -> None:
        """An n-gram language model with smoothing.

        Arguments:
          order: The order of the model (i.e., the value for n).
          k: The smoothing constant.
        """
        self.k = k
        super().__init__(order)

    def prob(self, context: tuple[str, ...], word: str) -> float:
        """Retrieve the (smoothed) probability of seeing a word given a context."""
        count = self.freq(context, word)
        total = self.total(context)
        vocab_size = len(self.vocab)
        return (count + self.k) / (total + self.k * vocab_size)

### 🤞 Test your code

The following cell sanity-checks your implementation:

In [42]:
with check_solutions() as check:
    model1 = NgramLM(2).fit(new_training_data)
    model2 = SmoothedLM(2, k=0).fit(new_training_data)
    check.equals(
        model1.perplexity(test_data),
        model2.perplexity(test_data),
        "With k=0, the perplexity scores should be exactly the same as with NgramLM",
    )
    model3 = SmoothedLM(2, k=1).fit(new_training_data)
    check(
        model3.perplexity(test_data) > model1.perplexity(test_data),
        "With k=1, the perplexity should be higher than without smoothing",
    )
    check.isclose(round(model3.perplexity(test_data)), 354)  # expected perplexity for k=1
    check(
        model3.perplexity(unseen_data) > 0,
        "With k=1, the model should be able to compute perplexity on the unseen test data without errors",
    )

## Task 7: Evaluate and compare models

Finally, let’s put everything together!  You should now train and compare $n$-gram models of different orders, on the new training data containing `<unk>`s, with different values for the smoothing constant $k$, and evaluate their perplexity on the unseen test data.

<div class="alert alert-info">

**Train and evaluate $n$-gram models and produce a table with the perplexity values!**

- For the order $n$, choose the same values as before: $n \in [1, 2, 3, 4]$
- For the smoothing constant $k$, try the following values: $k \in [0, 0.01, 0.1, 1]$
- Train all models on the training data with the unknown-word symbol, i.e. `new_training_data`.
- Compute perplexity on the unseen test data, i.e. `unseen_data`.  **For the cases where you cannot compute any perplexity scores, you can put "inf" in the results table.**
- Put all perplexity values from this evaluation in the `results` variable like shown below.  It is up to you how you want to do this: You can write code to populate the `results` list automatically; or you can simply run your code multiple times (with the different values for $n$ and $k$) and copy-and-paste the results into the `results` table.

</div>

In [44]:
results = [
    #         k=0     k=0.01  k=0.1   k=1
    ["n = 1", 0.00,   0.00,   0.00,   0.00],
    ["n = 2", 0.00,   0.00,   0.00,   0.00],
    ["n = 3", 0.00,   0.00,   0.00,   0.00],
    ["n = 4", 0.00,   0.00,   0.00,   0.00],
]

In [ ]:

orders = [1, 2, 3, 4]
k_values = [0, 0.01, 0.1, 1]


for i, order in enumerate(orders):
    for j, k in enumerate(k_values):
        try:
            if k == 0:
                model = NgramLM(order).fit(new_training_data)
            else:
                model = SmoothedLM(order, k=k).fit(new_training_data)
            
            pp = model.perplexity(unseen_data)
            results[i][j+1] = pp
        except Exception as e:
            
            results[i][j+1] = float("inf")

In [46]:
# This cells just displays the table in a nice way
tabulate(results, tablefmt="html", floatfmt=".2f", headers=["k = 0", "k = 0.01", "k = 0.1", "k = 1"])

,k = 0,k = 0.01,k = 0.1,k = 1
n = 1,322.35,322.36,322.47,323.98
n = 2,inf,207.79,264.45,567.48
n = 3,inf,874.33,1433.89,2966.69
n = 4,inf,2810.84,3677.96,5111.63


Finally, once you have all results, you should write a short paragraph on your observations.

<div class="alert alert-info">

**Answer the following question: Why and how does the smoothing constant influence the model’s perplexity?**  Provide an explanation based on your understanding of what smoothing does to the distribution of the probability mass.

</div>


When k = 0, the model gives infinite perplexity which means 0 probability. This means no event has probability 0.

As k increases, perplexity becomes finite and unseen n grams get non-zero probabilities. The model is less sure and assigns a higher mass to more outcomes.

The smoothing constant k introduces bias toward a more uniform distribution while reducing variance caused by sparse data. The best performance occurs when these 2 effects are balanced, which is why an intermediate value of k works better than either no/heavy smoothing.



- - -

**Congratulations on finishing this lab! 👍**

<div class="alert alert-info">

<strong>Before you submit...</strong>

➡️ Make sure you have filled in all cells that say **`YOUR CODE HERE`** or **YOUR ANSWER HERE**.

➡️ **Make sure the notebook runs from start to finish** without errors, for example by clicking the "⏩" button in the toolbar, or via "Run$\rightarrow$Restart Kernel and Run All Cells..." in the menu.  If this is successful, you should see a green box appear below!

</div>

In [47]:
from liu_lt import success
success("Notebook runs from start to finish!")